In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D
from scipy import stats

from esda.moran import Moran
from libpysal.weights import KNN
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import xskillscore as xs
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
import re

from collections import defaultdict
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 4
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 16, number of workers: 4, processes: False


2026-06-12 14:44:19,103 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:44:33,967 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:44:44,930 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:00,551 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:11,163 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:19,461 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:33,824 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:35,377 - distributed.utils_perf - WARNING - full garbage collections took

In [3]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jwille//proxy/39913/status,
Dashboard: /user/jwille//proxy/39913/status,Workers: 4
Total threads: 64,Total memory: 3.64 TiB
Status: running,Using processes: False
Comm: inproc://129.132.32.66/3158414/1,Workers: 4
Dashboard: /user/jwille//proxy/39913/status,Total threads: 64
Started: Just now,Total memory: 3.64 TiB
Comm: inproc://129.132.32.66/3158414/4,Total threads: 16
Dashboard: /user/jwille//proxy/36165/status,Memory: 0.91 TiB
Nanny: None,


## IFS deep on

In [ ]:
#IFS deep on land
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/daily_precip')
files = sorted([str(f) for f in data_path.glob('IFS_2.8-FESOM_5_daily_precip_*.nc')])[:]
ifs_on = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
indexer = ((ifs_on.lat>=latitude[0]) & (ifs_on.lat<=latitude[1])).compute()
ifs_on = ifs_on.where(indexer,drop=True)
ifs_on_sum = ifs_on.sum(dim='time').load()
ifs_on_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_2.8km_on_30-60N_precip_total_sum.nc')






## IFS deep off

In [ ]:
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/daily_precip')
files = sorted([str(f) for f in data_path.glob('IFS_2.8-FESOM_5_deep_off_daily_precip_*.nc')])[:]
ifs_off = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
indexer = ((ifs_off.lat>=latitude[0]) & (ifs_off.lat<=latitude[1])).compute()
ifs_off = ifs_off.where(indexer,drop=True)
ifs_off_sum = ifs_off.sum(dim='time').load()
ifs_off_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_2.8km_off_30-60N_precip_total_sum.nc')


## IFS cycle3

In [ ]:
data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/daily_precip')
files = sorted([str(f) for f in data_path.glob('daysum_IFS_4.4-FESOM_5_precip_30-60N_????.nc')])[:]
ifs_cycle3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
ifs_cycle3_sum = ifs_cycle3.resample(time="YS").sum().mean(dim='time')
ifs_cycle3_sum = ifs_cycle3_sum.load()
ifs_cycle3.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle3_30-60N_precip_total_sum.nc')



## IFS cycle4

In [ ]:
months = [5,6,7,8,9,10]

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/1990-2020/daily_precip')
files = sorted([str(f) for f in data_path.glob('30-60N_IFS_9-FESOM_5_daily_precip_????.nc')])[:]
ifs_cycle4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
ifs_cycle4 = ifs_cycle4.where(ifs_cycle4.time.dt.month.isin(months), drop=True)

ifs_cycle4_sum = ifs_cycle4.resample(time="YS").sum().mean(dim='time')
ifs_cycle4_sum = ifs_cycle4_sum.load()
ifs_cycle4_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_1990-2020_30-60N_precip_total_sum.nc')







## IFS cycle4 future

In [6]:
years = np.arange(2021,2050,1)
months = [5,6,7,8,9,10]

for year in years:
    print(year)
    ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/30-60N_IFS_9-FESOM_5_daily_precip_'+str(year)+'.nc')['tp']
    ifs_cycle4 = ifs_cycle4.where(ifs_cycle4.time.dt.month.isin(months), drop=True)
    ifs_cycle4_sum = ifs_cycle4.resample(time="YS").sum().mean(dim='time')
    ifs_cycle4_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_'+str(year)+'_30-60N_precip_total_sum.nc')

2021
2022
2023
2024
2025
2026
2027
2028
2029
2030
2031
2032
2033
2034
2035
2036
2037
2038
2039
2040
2041
2042
2043
2044
2045
2046
2047
2048
2049


## ICON cycle3

In [ ]:
data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/daily_precip')
files = sorted([str(f) for f in data_path.glob('daysum_ICON_ngc3028_hourly_precip_30-60N_????.nc')])[:]
icon_cycle3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['pr']
icon_cycle3_sum = icon_cycle3.resample(time="YS").sum().mean(dim='time')
icon_cycle3_sum = icon_cycle3_sum.load()
icon_cycle3_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle3_30-60N_precip_total_sum.nc')




## ICON cycle4

In [ ]:
months = [5,6,7,8,9,10]

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/daily_precip/temp')
files = sorted([str(f) for f in data_path.glob('30-60N_ICON_ngc4008_daily_precip_????.nc')])[:]
icon_cycle4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['pr']
icon_cycle4 = icon_cycle4.where(icon_cycle4.time.dt.month.isin(months), drop=True)

icon_cycle4_sum = icon_cycle4.resample(time="YS").sum().mean(dim='time')
icon_cycle4_sum = icon_cycle4_sum.load()
icon_cycle4_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle4_30-60N_precip_total_sum.nc')



## ICON cycle4 future

In [5]:
years = np.arange(2021,2050,1)
months = [5,6,7,8,9,10]

for year in years:
    print(year)
    ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/30-60N_IFS_9-FESOM_5_daily_precip_'+str(year)+'.nc')['tp']
    ifs_cycle4 = ifs_cycle4.where(ifs_cycle4.time.dt.month.isin(months), drop=True)
    ifs_cycle4_sum = ifs_cycle4.resample(time="YS").sum().mean(dim='time')
    ifs_cycle4_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_'+str(year)+'_30-60N_precip_total_sum.nc')

2021
2022
2023
2024
2025
2026
2027
2028
2029
2030
2031
2032
2033
2034
2035
2036
2037
2038
2039
2040
2041
2042
2043
2044
2045
2046
2047
2048
2049


## IMERG

In [ ]:
data_path = Path('/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/QC/daysum')
files = sorted([str(f) for f in data_path.glob('30-60N_daysum_QC_IMERG_V07B_final_????.nc4')])[:]
imerg = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['precipitation']
imerg_sum = imerg.resample(time="YS").sum().mean(dim='time')
imerg_sum = imerg_sum.load()
imerg_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/imerg_30-60N_precip_total_sum.nc')





## CESM2

In [7]:
# Change this to your directory path
model = 'CESM2'
#Source of CESM2 data /net/atmos/data/CESM2-LE/day/PRECT
#directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_past/temp'
directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_future'

# List all file names in the directory (adjust the filter as needed, e.g., for .nc files)
files = [f for f in os.listdir(directory) if f.endswith('.nc')]

# Create a dictionary to group files by simulation number
sim_files = defaultdict(list)


pattern = re.compile(r'LE2-1251\.(\d+)\.cam')

for file in files:
    match = pattern.search(file)
    if match:
        sim_num = match.group(1)
        sim_files[sim_num].append(file)
    else:
        print("Filename did not match expected pattern:", file)
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    # Here you can order or sort the files if the year range matters,
    # e.g., sorting by the start date in the filename (assuming YYYYMMDD is at a fixed position).
    # For example:
    file_list = sorted(file_list)

Processing simulation 013 with 9 files:
Processing simulation 011 with 9 files:
Processing simulation 014 with 9 files:
Processing simulation 012 with 9 files:
Processing simulation 015 with 9 files:
Processing simulation 016 with 9 files:
Processing simulation 017 with 9 files:
Processing simulation 018 with 9 files:
Processing simulation 019 with 9 files:
Processing simulation 020 with 9 files:


In [9]:
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    latitude = [30,60]
    months = [5,6,7,8,9,10]
    results_temporal = []
    
    file_list = sorted(file_list)
    # Create a full file path list
    full_paths = [os.path.join(directory, f) for f in file_list]

    #print(full_paths)
    dataset = xr.open_mfdataset(full_paths, engine='netcdf4',combine='by_coords')['PRECT']
    dataset = dataset * 86400 * 1000
    #dataset = dataset.sel(time=slice('1850', '2015'))  # select time range 
    dataset = dataset.sel(time=slice('2021', '2049'))  # select time range 
    dataset = dataset.where(dataset.time.dt.month.isin(months), drop=True)

    indexer = ((dataset.lat>=latitude[0]) & (dataset.lat<=latitude[1]))
    dataset = dataset.where(indexer,drop=True)
    dataset = dataset.convert_calendar('noleap')
 
    dataset_sum = dataset.resample(time="YS").sum()#.mean(dim='time')
    dataset_sum = dataset_sum.load()



    
    dataset_sum.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/cesm2_2021-2049_'+sim+'_30-60N_precip_total_sum.nc')



    
    #cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_2014-2049_'+sim+'_30-60N_temporal_corr.nc')




Processing simulation 013 with 9 files:
Processing simulation 011 with 9 files:
Processing simulation 014 with 9 files:
Processing simulation 012 with 9 files:
Processing simulation 015 with 9 files:


2026-06-12 14:45:29,221 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:30,594 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:45:32,192 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)


Processing simulation 016 with 9 files:


2026-06-12 14:47:37,347 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-06-12 14:47:38,800 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)


Processing simulation 017 with 9 files:


...

2026-06-12 14:51:53,800 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:51:55,574 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)


Processing simulation 019 with 9 files:


2026-06-12 14:54:03,376 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:54:05,224 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)


Processing simulation 020 with 9 files:


2026-06-12 14:56:08,061 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:56:11,776 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-06-12 14:56:13,625 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
